<div style="font-family:monospace; font-size:15px; line-height:1.5; border-top: 1px solid black; border-bottom: 1px solid black; padding: 10px; text-align: center;">
    <strong>CONVOLUTION NEURAL OPERATOR -  DIMENSIONALITY REDUCTION</strong><br>
    <strong></strong> SIVA VIKNESH<br>
    <strong></strong> siva.viknesh@sci.utah.edu / sivaviknesh14@gmail.com<br>
    <strong></strong> SCIENTIFIC COMPUTING & IMAGING INSTITUTE, UNIVERSITY OF UTAH, SALT LAKE CITY, UTAH, USA<br>
</div>

In [ ]:
# ****** IMPORTING THE NECESSARY LIBRARIES
import os
import torch
import random
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.parameter import Parameter
import vtk
from vtk.util.numpy_support import vtk_to_numpy 
import matplotlib.pyplot as plt

In [ ]:
processor = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("AVAILABLE PROCESSOR:", processor, '\n')

In [ ]:
# Set random seed for reproducibility
seed = 15
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE PARAMETERS
</div>

In [ ]:
# ***** PARAMETERS FOR THE NEURAL NETWORK
input_var    = 3                                        # INPUT VARIABLES {a(x, y), x, y}
x_dim        = 256
y_dim        = 256

# ***** HYPERPARAMETERS FOR THE NEURAL NETWORK
neurons      = 128
epochs       = 70
batchsize    = 30 
in_channels  = 1                       # NO. OF INPUT CHANNELS
out_channels = 1                       # NO. OF OUTPUT CHANNELS

learn_rate   = 1e-2
step_epoch   = 5
decay_rate   = 0.750

# ***** FILENAMES TO READ & WRITE THE DATA
mesh         = "Grid_256_0.vtk"
data_file    = "Grid_256_"
Nfile        = 1000                                 # NO. OF TIME INSTANTS

# ***** LOCATION TO READ AND WRITE THE DATA
pwd          = os.getcwd()
os.chdir("../../")
directory    = os.getcwd()
path         = directory + "/Flow_Data/"
mesh         = path + mesh
data_file    = path + data_file

# ***** NORMALISATION OF FLOW VARIABLES
xscale       = 10.0
yscale       = 5.0
vort_max     = 4.4

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    READING THE DATA FILES
</div>


In [ ]:
''' ***************************************** MESH FILE ****************************************************** '''
print ("*"*85)
print ('READING THE MESH FILE: ', mesh[len(directory)+1:])
reader = vtk.vtkStructuredPointsReader()
reader.SetFileName(mesh)
reader.Update()
data = reader.GetOutput()
n_points = data.GetNumberOfPoints()
print ('NO. OF GRID POINTS IN THE MESH:', n_points)
x_vtk_mesh = np.zeros((n_points,1))
y_vtk_mesh = np.zeros((n_points,1))
for i in range(n_points):
    pt_iso  =  data.GetPoint(i)
    x_vtk_mesh[i] = pt_iso[0] 
    y_vtk_mesh[i] = pt_iso[1]
    
x  = np.reshape(x_vtk_mesh, (x_dim, y_dim))/xscale 
y  = np.reshape(y_vtk_mesh, (x_dim, y_dim))/yscale

print ("SHAPE OF X:",   x.shape)
print ("SHAPE OF Y:",   y.shape)

print ("*"*85)

''' *************************************** FLOW FIELD FILE *************************************************** '''
fieldname1  = 'u_vel'                                                      # FIELD NAME FOR VTK FILES
fieldname2  = 'v_vel'
fieldname3  = 'vort_z'

#u_vel   = np.zeros((Nfile, x_dim, y_dim))
#v_vel   = np.zeros((Nfile, x_dim, y_dim))
vort    = np.zeros((Nfile, x_dim, y_dim))

for i in range(Nfile):
    file_name = data_file + str(i) + ".vtk"
    print ('READING THE DATA FILE: ', file_name[len(directory)+1:])
    reader = vtk.vtkStructuredPointsReader()
    reader.SetFileName(file_name)
    reader.Update()
    data = reader.GetOutput()
    #pointData1 = data.GetPointData().GetArray(fieldname1)
    #pointData2 = data.GetPointData().GetArray(fieldname2)    
    pointData3 = data.GetPointData().GetArray(fieldname3)    
    #u_vel   [i, :, :] = np.reshape(vtk_to_numpy(pointData1), (x_dim, y_dim))
    #v_vel   [i, :, :] = np.reshape(vtk_to_numpy(pointData2), (x_dim, y_dim))
    vort    [i, :, :] = np.reshape(vtk_to_numpy(pointData3), (x_dim, y_dim))
    
    print ("*"*85)
vort = vort / vort_max

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    DEFINING THE CLASS FUNCTIONS
</div>


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class Encoder(nn.Module):
    def __init__(self, in_channels):
        super(Encoder, self).__init__()
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.enc5 = DoubleConv(512, 1024)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        enc1 = self.enc1(x)                    # Size  = N*N
        enc2 = self.enc2(self.pool(enc1))      # Size  = N/2*N/2
        enc3 = self.enc3(self.pool(enc2))      # Size  = N/4*N/4
        enc4 = self.enc4(self.pool(enc3))      # Size  = N/8*N/8
        enc5 = self.enc5(self.pool(enc4))      # Size  = N/16*N/16
        return enc1, enc2, enc3, enc4, enc5

class Decoder(nn.Module):
    def __init__(self, out_channels):
        super(Decoder, self).__init__()
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, enc1, enc2, enc3, enc4, enc5):
        dec4 = self.up4(enc5)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)
        
        dec3 = self.up3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)
        
        dec2 = self.up2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)
        
        dec1 = self.up1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)
        
        return self.out_conv(dec1)

class CNO(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.encoder = Encoder(in_channels)
        self.decoder = Decoder(out_channels)

    def forward(self, x):
        enc1, enc2, enc3, enc4, enc5 = self.encoder(x)
        output = self.decoder(enc1, enc2, enc3, enc4, enc5)
        return output, enc1, enc2, enc3, enc4, enc5

def reproducible_split(dataset, train_size, test_size, seed=1):
    generator = torch.Generator().manual_seed(seed)
    return random_split(dataset, [train_size, test_size], generator=generator)

def count_parameters(model):
    total_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            total_params += param.numel()
    return total_params


In [ ]:
# MESH SPATIAL DATA
x_grid = torch.Tensor(x).reshape(1, x_dim, y_dim, 1)
y_grid = torch.Tensor(y).reshape(1, x_dim, y_dim, 1)

# FUNCTION MAPPING DATA
vort_input   = torch.Tensor(vort).to(processor)
vort_output  = torch.Tensor(vort).to(processor)

# SORTING THE TRAINING DATA
shape         = vort_input.size()
total_samples = shape[0]

train_size    = int(0.8 * total_samples)
test_size     = total_samples - train_size

vort_input_reshaped  = vort_input.reshape  (total_samples, 1, x_dim, y_dim)
vort_output_reshaped = vort_output.reshape (total_samples, 1, x_dim, y_dim)

dataset               = TensorDataset(vort_input_reshaped, vort_output_reshaped)
vort_train, vort_test = random_split(dataset, [train_size, test_size], generator=torch.manual_seed(seed))

train_loader   = DataLoader(vort_train, batch_size=batchsize, shuffle=True)
test_loader    = DataLoader(vort_test,  batch_size=10, shuffle=False)
plot_loader    = DataLoader(dataset, batch_size=batchsize, shuffle=False)

model = CNO (in_channels, out_channels).to(processor)
print("Encoder trainable parameters:", count_parameters(model))

optimizer  = optim.Adam(model.parameters(), lr=learn_rate, betas = (0.9,0.99),eps = 10**-15)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=step_epoch, gamma=decay_rate)

In [ ]:
Loss_Data   = torch.empty(size=(epochs+1, 1))
loss_func   = nn.MSELoss()

for epoch in range (epochs+1):
    total_loss = 0.0
    for batch_idx, (w_in, w_out) in enumerate(train_loader):
        output , en1, en2, en3, en4, en5 = model (w_in)
        batch_loss          = loss_func (output, w_out)

        optimizer.zero_grad()
        batch_loss.backward()

        with torch.no_grad():
            optimizer.step()

        total_loss += batch_loss.detach()  

    N = batch_idx + 1
    Loss_Data   [epoch] = total_loss/N
        
    print('TOTAL AVERAGE LOSS, [EPOCH =', epoch,']')
    print('LOSS          :', Loss_Data[epoch].item())
    print('LEARNING RATE :', optimizer.param_groups[0]['lr'])
    print ("*"*85)
    
    scheduler.step()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
    SAVING THE FILES
</div>

In [ ]:
os.chdir(pwd)
torch.save(model.state_dict(),  "CNO_MODEL.pt" )
torch.save(Loss_Data  [0:epoch],"CNO_Loss.pt"  )

In [ ]:
os.chdir(pwd)
model.load_state_dict(torch.load("CNO_MODEL.pt", weights_only=True))
loss_func   = nn.MSELoss()

<div style="text-align: center; font-weight: bold; font-size: 1em;">
        TESTING DATA
</div>


In [ ]:
# TEST ERROR OF CNO
model.eval()

total_loss = 0.0
for batch_idx, (w_in, w_out) in enumerate(test_loader):
    output , en1, en2, en3, en4, en5 = model (w_in)
    batch_loss          = loss_func (output, w_out)

    with torch.no_grad():
        total_loss += batch_loss.detach()  

N = batch_idx + 1
Test_Loss = total_loss/N

print('TOTAL AVERAGE TESTING LOSS:', Test_Loss.item())

In [ ]:
test_loader    = DataLoader(vort_test, batch_size=3, shuffle=False)
for batch_idx, (w_in, w_out) in enumerate(test_loader):
    output , en1, en2, en3, en4, en5 = model (w_in)
    absolute_error  = torch.abs(output - w_out)

fig1, ax1 = plt.subplots(1, 3, figsize=(18, 5))

titles = ['Input Velocity', 'Output Velocity', 'Absolute Error']
data_list = [w_in, w_out, absolute_error]

for i in range(3):
    pcm = ax1[i].contourf(data_list[i][0, 0].detach().cpu().numpy(), cmap='jet')  
    ax1[i].set_title(titles[i])
    ax1[i].set_xlabel('X')
    ax1[i].set_ylabel('Y')
    ax1[i].set_box_aspect(aspect=1)
    fig1.colorbar(pcm, ax=ax1[i], orientation='horizontal')

plt.tight_layout()
plt.savefig("Fig_CNO_Error.png")
plt.show()

encoder_outputs = [en1, en2, en3, en4, en5]
encoder_titles = ['Encoder 1', 'Encoder 2', 'Encoder 3', 'Encoder 4', 'Encoder 5']

for i in range(5):
    fig, ax = plt.subplots(figsize=(5, 5))  # Create a separate figure for each encoder
    pcm = ax.contourf(encoder_outputs[i][0, 0].detach().cpu().numpy(), cmap='jet')  
    ax.set_title(encoder_titles[i])
    
    # Disable labels and ticks
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    ax.set_box_aspect(aspect=1)
    fig.colorbar(pcm, ax=ax, orientation='horizontal')

    plt.tight_layout()
    plt.savefig(f"Fig_CNO_Encoder_{i+1}.png")  # Save each figure separately
    plt.close(fig)

In [ ]:
def save_to_tecplot(filename, tensor, title, varname="Value"):
    """
    Save a 2D tensor (torch.Tensor) to a Tecplot-compatible ASCII .dat file.
    """
    # Convert to NumPy and squeeze
    np_data = torch.squeeze(tensor).detach().cpu().numpy()
    
    # Get dimensions
    ny, nx = np_data.shape
    
    # Generate coordinate mesh (assuming uniform grid in [0, 1] x [0, 1])
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)

    # Flatten everything
    X_flat = X.flatten()
    Y_flat = Y.flatten()
    val_flat = np_data.flatten()
    
    # Write to file
    with open(filename, 'w') as f:
        f.write(f'TITLE = "{title}"\n')
        f.write(f'VARIABLES = "X", "Y", "{varname}"\n')
        f.write(f'ZONE T="{title}", I={nx}, J={ny}, F=POINT\n')
        for i in range(len(X_flat)):
            f.write(f"{X_flat[i]:.6f} {Y_flat[i]:.6f} {val_flat[i]:.6e}\n")
            
# Example usage
save_to_tecplot("input_velocity.dat", w_in[0], "Input Velocity", "U_in")
save_to_tecplot("encoded_bottleneck_1.dat", en1[0, 0], "Encoded Bottleneck", "U_enc")
save_to_tecplot("encoded_bottleneck_2.dat", en2[0, 0], "Encoded Bottleneck", "U_enc")
save_to_tecplot("encoded_bottleneck_3.dat", en3[0, 0], "Encoded Bottleneck", "U_enc")
save_to_tecplot("encoded_bottleneck_4.dat", en4[0, 0], "Encoded Bottleneck", "U_enc")
save_to_tecplot("output_velocity.dat", output[0], "Output Velocity", "U_out")
save_to_tecplot("absolute_error.dat", absolute_error[0], "Absolute Error", "AbsErr")